### *從Windows\Fonts複製微軟正黑體字型檔msjh.ttc至這個檔案所在目錄
### *安裝下面模組
+ pip install moviepy  opencv-python  yt-dlp
+ 安裝<a href="https://imagemagick.org/index.php"> ImageMagic </a>
+ 加入環境變數 IMAGEMAGICK_BINARY
  
  <img src="https://tronclass.ntou.edu.tw/api/uploads/8381120/in-rich-content?created_at=2025-01-25T01:49:33Z" width="400px">

In [5]:
import numpy as np
import pyttsx3 
from moviepy import AudioFileClip
from moviepy import VideoFileClip, TextClip, CompositeVideoClip, CompositeAudioClip, concatenate_audioclips
from moviepy.video.tools.subtitles import SubtitlesClip
from moviepy.video.fx import MultiplySpeed
from moviepy.audio.fx import MultiplyVolume



class TexttoSpeech:
    def __init__(self):
        # Initialize the engine
            
        self.engine = pyttsx3.init()

        voices = self.engine.getProperty('voices')
        self.chinese_voice_id = None

        for v in voices:
            if "Chinese" in v.name or "ZH-TW" in v.id:
                self.chinese_voice_id = v.id
                break

        if self.chinese_voice_id:
            self.engine.setProperty('voice', self.chinese_voice_id)
        else:
            print("No Chinese voice found. Please install one in System Settings.")

    def text_to_speech(self,message):
        self.engine.say(message) 
    
    def text_to_mp3(self,message,mp3file):
        self.engine.save_to_file(message, mp3file)

    def runAndWait(self):
        self.engine.runAndWait()

        
ts = TexttoSpeech()

#腳本，格式是一行一句。
subtitles = '''    "這是作業一",
    "上面第一個row由左到右展示了原始的YouTube影片",
    "影像負片",
    "直方圖均衡化",
    "第二個row由左到右展示了原始的本地測試影片",
    "HSV色彩空間",
    "用pytorch做的馬賽克效果"'''
   
# 來源視訊，背景音樂，目標視訊檔名。    
source_video_filename     = "hw1_output.mp4" 
background_music_filename = "calm-background-for-video-121519.mp3" # google this mp3 and download it by yourself
target_video_with_subtitle= "homework_1_test_video_subtitle.mp4"

lines = [msg.strip() for msg in subtitles.split('\n') if len(msg)>0]
total_lines = len(lines)
speech= []
#念每一句旁白。
for i,msg in enumerate(lines):
    print(f'generate audio clip for {msg} ({i+1}/{total_lines})')
    ts.text_to_mp3(msg,'voice-msg-{:d}.mp3'.format(i))
#ts.engine.runAndWait()

### 根據每句旁白時間長度，計算每句旁白在視訊裡的起始與結束時間
speech = [AudioFileClip('voice-msg-{:d}.mp3'.format(i)) for i, _ in enumerate(lines)]   
duration = np.array([0]+[s.duration for s in speech])   
cumduration = np.cumsum(duration)
total_duration = int((cumduration[-1]+7)/8)*8    

print(total_duration)



clip = VideoFileClip(source_video_filename)

#調整目標視訊播放速度，使得目標視訊播放時間比念完全部旁白長一點。
speed_factor = clip.duration / total_duration
clip = clip.with_effects([MultiplySpeed(speed_factor)])

# 產生旁白字幕，注意msjh.ttc字型檔要在這個程式所在目錄。
generator = lambda txt: TextClip(text=txt, font='msjhbd.ttc', font_size=48, color='white', bg_color=(10,10,10,150), method='caption',size=(clip.w, 64))
subs_data = [((cumduration[i],cumduration[i+1]),s) for i,s in enumerate(lines)]
subtitles = SubtitlesClip(subs_data, make_textclip=generator)

#產生有字幕的目標視訊。
final_clip = CompositeVideoClip([clip, subtitles.with_position(("center","bottom"))])

#背景音樂，只擷取目標視訊長度片段，並將音量調小。
background_audio = AudioFileClip(background_music_filename)
baudio1 = background_audio.subclipped(background_audio.duration-total_duration).with_effects([MultiplyVolume(0.1)])

#將目標視訊的音訊設為混合來源視訊音訊、背景音樂、旁白音訊的音訊。
final_clip = final_clip.with_audio(CompositeAudioClip([baudio1,concatenate_audioclips(speech)]))

#輸出至目標視訊檔。
final_clip.write_videofile(target_video_with_subtitle)



generate audio clip for "這是作業一", (1/7)
generate audio clip for "上面第一個row由左到右展示了原始的YouTube影片", (2/7)
generate audio clip for "影像負片", (3/7)
generate audio clip for "直方圖均衡化", (4/7)
generate audio clip for "第二個row由左到右展示了原始的本地測試影片", (5/7)
generate audio clip for "HSV色彩空間", (6/7)
generate audio clip for "用pytorch做的馬賽克效果" (7/7)
24
MoviePy - Building video homework_1_test_video_subtitle.mp4.
MoviePy - Writing audio in homework_1_test_video_subtitleTEMP_MPY_wvf_snd.mp3


MoviePy - Done.
MoviePy - Writing video homework_1_test_video_subtitle.mp4



MoviePy - Done !
MoviePy - video ready homework_1_test_video_subtitle.mp4
